# Session 2 — Triggering the Retrain Pipeline

**Goal:** Manually trigger the retrain job the same way a CI/CD system would.

The instructor has pre-deployed the `workshop_retrain_churn` job using:
```bash
databricks bundle deploy --target prod
```

You'll trigger a run via the Databricks SDK and watch it execute in real time.

In [0]:
%run ../utils/config

In [0]:
from databricks.sdk import WorkspaceClient
import time

w = WorkspaceClient()

# Find the retrain job by name
jobs_list = list(w.jobs.list(name=f"[dev {schema}] workshop_retrain_churn"))
if not jobs_list:
    raise ValueError(f"Job '[dev {schema}] workshop_retrain_churn' not found. Has the instructor deployed the bundle?")

job = jobs_list[0]
print(f"Found job: {job.settings.name}")
print(f"Job ID   : {job.job_id}")

In [0]:
# Trigger a run
run_response = w.jobs.run_now(job_id=job.job_id)
run_id = run_response.run_id
print(f"Run triggered!")
print(f"Run ID  : {run_id}")
print(f"View at : {w.config.host}#job/{job.job_id}/run/{run_id}")

## Navigate to the Job Run

1. Click the URL printed above (or go to **Workflows → Jobs** in the left sidebar)
2. Find `workshop_retrain_churn` and click the running run
3. Observe:
   - The **task dependency graph** — tasks turn green as they complete
   - Click any task to see its **log output**
   - The training task will log a link to the MLflow experiment

> The full run takes **4-8 minutes**.

In [0]:
# Poll until the job completes (optional — you can also just watch the UI)
print("Polling for completion (press Stop to cancel and watch in the UI)...")
while True:
    run_state = w.jobs.get_run(run_id=run_id)
    state = run_state.state.life_cycle_state.value
    result = run_state.state.result_state
    print(f"  State: {state}" + (f" | Result: {result.value}" if result else ""))
    
    if state in ["TERMINATED", "SKIPPED", "INTERNAL_ERROR"]:
        break
    time.sleep(20)

if run_state.state.result_state.value == "SUCCESS":
    print("\n✓ Retrain pipeline completed successfully!")
    print("  The @champion model alias has been updated.")
else:
    print(f"\n⚠️  Run finished with: {run_state.state.result_state.value}")
    print("  Check the task logs for details.")

## Discussion

- Which task took the longest? Why?
- What happens to the job if the data quality check fails?
- How would you schedule this job to run automatically every week?

➡️ Next: `03_deployment_gate.ipynb`